<a href="https://colab.research.google.com/github/MohammedShahad7/Data-Science-/blob/main/Handling%20Missing%20Data%20for%20Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)

n = 200

df = pd.DataFrame({
    "Ride_ID": [f"RIDE_{i+1}" for i in range(n)],
    "Ride_Distance": np.random.normal(8, 2.5, n).round(2),
    "Fare_Amount": np.random.normal(250, 80, n).round(2),
    "Driver_Rating": np.random.choice([3, 4, 5, np.nan], size=n, p=[0.2, 0.2, 0.12, 0.48]),
    "Vehicle_Type": np.random.choice(["Sedan", "SUV", "Mini", np.nan], size=n, p=[0.3, 0.3, 0.34, 0.06]),
    "Outside_Temperature": np.random.normal(30, 4, n).round(1),
    "Ride_Time": pd.date_range(start="2023-01-01", periods=n, freq="H")
})

# Introduce additional missingness
df.loc[df.sample(frac=0.03).index, "Ride_Distance"] = np.nan
df.loc[df.sample(frac=0.01).index, "Fare_Amount"] = np.nan
df.loc[df.sample(frac=0.12).index, "Outside_Temperature"] = np.nan

df.head()

/tmp/ipykernel_2156/2997814897.py:15: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  "Ride_Time": pd.date_range(start="2023-01-01", periods=n, freq="H")


,Ride_ID,Ride_Distance,Fare_Amount,Driver_Rating,Vehicle_Type,Outside_Temperature,Ride_Time
0,RIDE_1,9.24,278.62,5.0,Sedan,31.2,2023-01-01 00:00:00
1,RIDE_2,7.65,294.86,4.0,Mini,23.2,2023-01-01 01:00:00
2,RIDE_3,9.62,NaN,3.0,Sedan,24.6,2023-01-01 02:00:00
3,RIDE_4,11.81,334.30,NaN,Sedan,33.0,2023-01-01 03:00:00
4,RIDE_5,7.41,139.79,NaN,Sedan,30.7,2023-01-01 04:00:00


In [23]:


# Percentage of missing values for every column
percent_missing = df.isnull().mean() * 100

# Create strategy table
strategy_table = pd.DataFrame({
    'Column': df.columns,
    'Data_Type': df.dtypes.astype(str).values,
    'Percent_Missing': percent_missing.values
})
def assign_strategy(row):
    col = row["Column"]
    dtype = row["Data_Type"]
    missing = row["Percent_Missing"]

    # No missing values
    if missing == 0:
        return "None"

    # High missingness
    if missing >= 40:
        return "Flag_and_Zero"

    # Time-series columns
    if "time" in col.lower() or "temp" in col.lower():
        return "Forward_Fill"

    # Numeric columns
    if dtype in ["int64", "float64"]:
        return "Median"

    # Categorical columns
    return "Mode"

strategy_table["Chosen_Strategy"] = strategy_table.apply(assign_strategy, axis=1)

strategy_table

,Column,Data_Type,Percent_Missing,Chosen_Strategy
0,Ride_ID,object,0.0,None
1,Ride_Distance,float64,3.0,Median
2,Fare_Amount,float64,1.0,Median
3,Driver_Rating,float64,43.0,Flag_and_Zero
4,Vehicle_Type,object,0.0,None
5,Outside_Temperature,float64,12.0,Forward_Fill
6,Ride_Time,datetime64[ns],0.0,None


In [32]:
# Impute Ride_Distance using Median
df['Ride_Distance'].fillna(df['Ride_Distance'].median())

# Impute Fare_Amount using Median
df['Fare_Amount'].fillna(df['Fare_Amount'].median())

# Impute Vehicle_Type using Mode
df['Vehicle_Type'].fillna(df['Vehicle_Type'].mode()[0])

# Sort dataset chronologically by Ride_Time
df = df.sort_values(by='Ride_Time')

# Forward-fill Outside_Temperature
df['Outside_Temperature'] = df['Outside_Temperature'].ffill()

# Create binary indicator for missing Driver_Rating
df['Driver_Rating_Missing'] = df['Driver_Rating'].isna().astype(int)

# Impute missing Driver_Rating values with 0
df['Driver_Rating'] = df['Driver_Rating'].fillna(0)

df

,Ride_ID,Ride_Distance,Fare_Amount,Driver_Rating,Vehicle_Type,Outside_Temperature,Ride_Time,Driver_Rating_Missing
0,RIDE_1,9.24,278.62,False,Sedan,31.2,2023-01-01 00:00:00,0
1,RIDE_2,7.65,294.86,False,Mini,23.2,2023-01-01 01:00:00,0
2,RIDE_3,9.62,NaN,False,Sedan,24.6,2023-01-01 02:00:00,0
3,RIDE_4,11.81,334.30,False,Sedan,33.0,2023-01-01 03:00:00,0
4,RIDE_5,7.41,139.79,False,Sedan,30.7,2023-01-01 04:00:00,0
...,...,...,...,...,...,...,...,...
195,RIDE_196,8.96,212.47,False,SUV,31.1,2023-01-09 03:00:00,0
196,RIDE_197,5.79,112.95,False,Mini,32.4,2023-01-09 04:00:00,0
197,RIDE_198,8.38,358.31,False,Mini,30.7,2023-01-09 05:00:00,0
198,RIDE_199,8.15,240.84,False,SUV,28.2,2023-01-09 06:00:00,0


In [39]:
# Average Fare_Amount for rides where Driver_Rating was missing
avg_missing = df[df['Driver_Rating_Missing'] == 1]['Fare_Amount'].mean()

# Average Fare_Amount for rides where Driver_Rating was provided
avg_non_missing = df[df['Driver_Rating_Missing'] == 0]['Fare_Amount'].mean()

# Create analysis_table
analysis_table = pd.DataFrame({
    'Group': ['Missing Rating', 'Non-Missing Rating'],
    'Average_Fare': [avg_missing, avg_non_missing],
    'Ride_Count': [
        (df['Driver_Rating_Missing'] == 1).sum(),
        (df['Driver_Rating_Missing'] == 0).sum()
    ]
})

print(analysis_table)

                Group  Average_Fare  Ride_Count
0      Missing Rating           NaN           0
1  Non-Missing Rating    256.626414         200
